# TfL Data Pipeline - Production Version
## London Mobility Intelligence Lakehouse

### Pipeline Flow:
* **Bronze Layer**: Raw JSON ingestion from TfL API
* **Silver Layer**: Parsed and structured data
* **Gold Layer**: Business aggregations and metrics
* **Alerts**: Service disruption detection

### Tables Created:
* tfl.bronze.line_status_bz
* tfl.bronze.arrivals_bz
* tfl.bronze.stop_point_arrivals_bz
* tfl.silver.line_status_sv
* tfl.silver.arrivals_sv
* tfl.gold.line_performance_gd
* tfl.gold.station_activity_gd
* tfl.gold.hourly_patterns_gd
* tfl.gold.status_history_gd
* tfl.gold.alerts_log_gd

In [0]:
import json
import time
import requests
from datetime import datetime, timezone
from functools import reduce

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, lit, current_timestamp, get_json_object, avg, count, min, max, stddev, hour, date_format, round

spark = SparkSession.builder.getOrCreate()

In [0]:
TFL_BASE_URL = "https://api.tfl.gov.uk"
APP_ID = ""
APP_KEY = ""
AUTH_PARAMS = {"app_id": APP_ID, "app_key": APP_KEY}

INGEST_DATE = "2026-02-07"
CATALOG = "tfl"
BRONZE_SCHEMA = "bronze"

In [0]:
def fetch_tfl_api(endpoint: str, params: dict = None) -> list:
    """
    Fetch data from TfL API endpoint.
    
    Args:
        endpoint: API endpoint path
        params: Additional query parameters
    
    Returns:
        List of JSON records or empty list on failure
    """
    url = f"{TFL_BASE_URL}{endpoint}"
    query_params = {**AUTH_PARAMS, **(params or {})}
    
    try:
        response = requests.get(url, params=query_params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return []


def create_bronze_df(records: list, data_source: str, fetch_timestamp: str) -> DataFrame:
    """
    Convert JSON records to DataFrame with provenance columns.
    
    Args:
        records: List of JSON records from API
        data_source: Source identifier
        fetch_timestamp: ISO timestamp when API fetch began
    
    Returns:
        DataFrame with raw_json and provenance columns
    """
    if not records:
        return spark.createDataFrame(
            [], 
            schema="raw_json STRING, data_source STRING, fetch_timestamp_utc TIMESTAMP, ingest_date STRING"
        )
    
    df = spark.createDataFrame(
        [(json.dumps(record),) for record in records],
        schema="raw_json STRING"
    )
    
    df = df.withColumn("data_source", lit(data_source)) \
           .withColumn("fetch_timestamp_utc", lit(fetch_timestamp).cast("timestamp")) \
           .withColumn("ingest_date", lit(INGEST_DATE))
    
    return df


def write_to_bronze(df: DataFrame, table_name: str) -> None:
    """
    Write DataFrame to Bronze table.
    """
    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("ingest_date") \
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")

In [0]:
def ingest_line_status() -> None:
    """Ingest Line Status data for all tube, DLR, and Overground lines."""
    fetch_timestamp = datetime.now(timezone.utc).isoformat()
    modes = "tube,dlr,overground,elizabeth-line"
    endpoint = f"/Line/Mode/{modes}/Status"
    records = fetch_tfl_api(endpoint)
    df = create_bronze_df(records, data_source="line_status", fetch_timestamp=fetch_timestamp)
    write_to_bronze(df, table_name="line_status_bz")


def ingest_line_arrivals() -> None:
    """Ingest Line Arrivals data for key tube lines."""
    fetch_timestamp = datetime.now(timezone.utc).isoformat()
    lines = [
        "bakerloo", "central", "circle", "district", "hammersmith-city",
        "jubilee", "metropolitan", "northern", "piccadilly", "victoria",
        "waterloo-city", "elizabeth", "dlr"
    ]
    all_records = []
    for line in lines:
        endpoint = f"/Line/{line}/Arrivals"
        records = fetch_tfl_api(endpoint)
        all_records.extend(records)
    df = create_bronze_df(all_records, data_source="line_arrivals", fetch_timestamp=fetch_timestamp)
    write_to_bronze(df, table_name="arrivals_bz")


def ingest_stop_point_arrivals() -> None:
    """Ingest Stop Point Arrivals for major stations."""
    fetch_timestamp = datetime.now(timezone.utc).isoformat()
    stop_points = {
        "940GZZLUKSX": "King's Cross",
        "940GZZLUVIC": "Victoria",
        "940GZZLULST": "Liverpool Street",
        "940GZZLUWLO": "Waterloo",
        "940GZZLUPAD": "Paddington",
        "940GZZLUBNK": "Bank-Monument"
    }
    all_records = []
    for naptan_id, name in stop_points.items():
        endpoint = f"/StopPoint/{naptan_id}/Arrivals"
        records = fetch_tfl_api(endpoint)
        all_records.extend(records)
    df = create_bronze_df(all_records, data_source="stop_point", fetch_timestamp=fetch_timestamp)
    write_to_bronze(df, table_name="stop_point_arrivals_bz")

In [0]:
def setup_bronze_layer() -> None:
    """Create catalog and schema if they don't exist."""
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")


def run_bronze_ingestion() -> None:
    """Execute Bronze layer ingestion for all three sources."""
    setup_bronze_layer()
    ingest_line_status()
    ingest_line_arrivals()
    ingest_stop_point_arrivals()

## Silver Layer Transformations

In [0]:
def run_silver_transformations() -> None:
    """Execute Silver layer transformations."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
    
    # Line Status
    line_status_bz = spark.table("tfl.bronze.line_status_bz")
    line_status_sv = line_status_bz \
        .withColumn("line_id", get_json_object(col("raw_json"), "$.id")) \
        .withColumn("line_name", get_json_object(col("raw_json"), "$.name")) \
        .withColumn("mode_name", get_json_object(col("raw_json"), "$.modeName")) \
        .withColumn("status_severity", get_json_object(col("raw_json"), "$.lineStatuses[0].statusSeverity").cast("int")) \
        .withColumn("status_description", get_json_object(col("raw_json"), "$.lineStatuses[0].statusSeverityDescription")) \
        .withColumn("created", get_json_object(col("raw_json"), "$.created").cast("timestamp")) \
        .withColumn("modified", get_json_object(col("raw_json"), "$.modified").cast("timestamp")) \
        .select("line_id", "line_name", "mode_name", "status_severity", "status_description", 
                "created", "modified", "data_source", "fetch_timestamp_utc", "ingest_date")
    
    line_status_sv.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date").saveAsTable("tfl.silver.line_status_sv")
    
    # Arrivals (unified)
    arrivals_bz_all = spark.table("tfl.bronze.arrivals_bz").union(spark.table("tfl.bronze.stop_point_arrivals_bz"))
    arrivals_sv = arrivals_bz_all \
        .withColumn("prediction_id", get_json_object(col("raw_json"), "$.id")) \
        .withColumn("vehicle_id", get_json_object(col("raw_json"), "$.vehicleId")) \
        .withColumn("naptan_id", get_json_object(col("raw_json"), "$.naptanId")) \
        .withColumn("station_name", get_json_object(col("raw_json"), "$.stationName")) \
        .withColumn("line_id", get_json_object(col("raw_json"), "$.lineId")) \
        .withColumn("line_name", get_json_object(col("raw_json"), "$.lineName")) \
        .withColumn("platform_name", get_json_object(col("raw_json"), "$.platformName")) \
        .withColumn("direction", get_json_object(col("raw_json"), "$.direction")) \
        .withColumn("destination_naptan_id", get_json_object(col("raw_json"), "$.destinationNaptanId")) \
        .withColumn("destination_name", get_json_object(col("raw_json"), "$.destinationName")) \
        .withColumn("prediction_timestamp", get_json_object(col("raw_json"), "$.timestamp").cast("timestamp")) \
        .withColumn("time_to_station_seconds", get_json_object(col("raw_json"), "$.timeToStation").cast("int")) \
        .withColumn("current_location", get_json_object(col("raw_json"), "$.currentLocation")) \
        .withColumn("towards", get_json_object(col("raw_json"), "$.towards")) \
        .withColumn("expected_arrival", get_json_object(col("raw_json"), "$.expectedArrival").cast("timestamp")) \
        .withColumn("mode_name", get_json_object(col("raw_json"), "$.modeName")) \
        .select("prediction_id", "vehicle_id", "naptan_id", "station_name", "line_id", "line_name",
                "platform_name", "direction", "destination_naptan_id", "destination_name",
                "prediction_timestamp", "time_to_station_seconds", "current_location", "towards",
                "expected_arrival", "mode_name", "data_source", "fetch_timestamp_utc", "ingest_date")
    
    arrivals_sv.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date", "data_source").saveAsTable("tfl.silver.arrivals_sv")

## Gold Layer Aggregations

In [0]:
def run_gold_aggregations() -> None:
    """Execute Gold layer aggregations."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
    
    arrivals = spark.table("tfl.silver.arrivals_sv")
    line_status = spark.table("tfl.silver.line_status_sv")
    
    # Line Performance
    line_performance = arrivals \
        .groupBy("line_name", "line_id", "mode_name", "ingest_date") \
        .agg(
            count("*").alias("total_predictions"),
            avg("time_to_station_seconds").alias("avg_wait_time_seconds"),
            min("time_to_station_seconds").alias("min_wait_time_seconds"),
            max("time_to_station_seconds").alias("max_wait_time_seconds"),
            stddev("time_to_station_seconds").alias("stddev_wait_time_seconds"),
            count(col("vehicle_id").isNotNull()).alias("active_vehicles")
        ) \
        .withColumn("avg_wait_time_minutes", round(col("avg_wait_time_seconds") / 60, 2))
    
    line_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date").saveAsTable("tfl.gold.line_performance_gd")
    
    # Station Activity
    station_activity = arrivals \
        .groupBy("station_name", "naptan_id", "data_source", "ingest_date") \
        .agg(
            count("*").alias("total_arrivals"),
            count(col("line_name").isNotNull()).alias("distinct_lines_serving"),
            avg("time_to_station_seconds").alias("avg_wait_time_seconds")
        ) \
        .withColumn("avg_wait_time_minutes", round(col("avg_wait_time_seconds") / 60, 2))
    
    station_activity.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date", "data_source").saveAsTable("tfl.gold.station_activity_gd")
    
    # Hourly Patterns
    hourly_patterns = arrivals \
        .withColumn("hour_of_day", hour("prediction_timestamp")) \
        .withColumn("day_of_week", date_format("prediction_timestamp", "EEEE")) \
        .groupBy("line_name", "hour_of_day", "day_of_week", "ingest_date") \
        .agg(
            count("*").alias("arrival_count"),
            avg("time_to_station_seconds").alias("avg_wait_time_seconds")
        ) \
        .withColumn("avg_wait_time_minutes", round(col("avg_wait_time_seconds") / 60, 2))
    
    hourly_patterns.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date").saveAsTable("tfl.gold.hourly_patterns_gd")
    
    # Status History
    status_history = line_status \
        .groupBy("line_name", "line_id", "mode_name", "status_severity", "status_description", "ingest_date") \
        .agg(
            count("*").alias("status_snapshot_count"),
            min("fetch_timestamp_utc").alias("first_observed"),
            max("fetch_timestamp_utc").alias("last_observed")
        )
    
    status_history.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date").saveAsTable("tfl.gold.status_history_gd")

In [0]:
def run_alert_detection() -> int:
    """Detect and log service alerts. Returns count of active alerts."""
    alerts_df = spark.sql("""
        SELECT 
            current_timestamp() as alert_timestamp,
            p.line_name,
            p.mode_name,
            s.status_description,
            s.status_severity,
            p.avg_wait_time_minutes,
            p.active_vehicles,
            p.total_predictions,
            CASE 
                WHEN s.status_severity < 10 THEN 'Service Disruption'
                WHEN p.avg_wait_time_minutes > 25 THEN 'Excessive Wait Time'
                WHEN CAST(p.active_vehicles AS INT) = 0 THEN 'No Active Vehicles'
                ELSE 'Other'
            END as alert_type,
            CASE 
                WHEN s.status_severity < 10 THEN 'High'
                WHEN CAST(p.active_vehicles AS INT) = 0 THEN 'High'
                WHEN p.avg_wait_time_minutes > 25 THEN 'Medium'
                ELSE 'Low'
            END as alert_priority,
            p.ingest_date,
            s.first_observed
        FROM tfl.gold.line_performance_gd p
        INNER JOIN tfl.gold.status_history_gd s 
            ON p.line_name = s.line_name 
            AND p.ingest_date = s.ingest_date
        WHERE s.status_severity < 10 
            OR p.avg_wait_time_minutes > 25 
            OR CAST(p.active_vehicles AS INT) = 0
        ORDER BY 
            CASE 
                WHEN s.status_severity < 10 THEN 1
                WHEN CAST(p.active_vehicles AS INT) = 0 THEN 2
                WHEN p.avg_wait_time_minutes > 25 THEN 3
                ELSE 4
            END,
            p.avg_wait_time_minutes DESC
    """)
    
    alerts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("ingest_date").saveAsTable("tfl.gold.alerts_log_gd")
    
    return alerts_df.count()

## Full Pipeline Execution

In [0]:
def run_full_pipeline() -> dict:
    """Execute full TfL data pipeline and return summary."""
    start_time = time.time()
    
    # Bronze
    run_bronze_ingestion()
    bronze_time = time.time() - start_time
    
    # Silver
    silver_start = time.time()
    run_silver_transformations()
    silver_time = time.time() - silver_start
    
    # Gold
    gold_start = time.time()
    run_gold_aggregations()
    gold_time = time.time() - gold_start
    
    # Alerts
    alert_start = time.time()
    alert_count = run_alert_detection()
    alert_time = time.time() - alert_start
    
    total_time = time.time() - start_time
    
    return {
        "bronze_time": bronze_time,
        "silver_time": silver_time,
        "gold_time": gold_time,
        "alert_time": alert_time,
        "total_time": total_time,
        "alert_count": alert_count
    }


# Execute pipeline
summary = run_full_pipeline()